# Unit Tests — Schema Validation

Validates that each Medallion layer exposes exactly the columns it promises.

| Layer | What is tested |
|-------|---------------|
| **Bronze** | Generator output has all required columns; no undocumented columns exist |
| **Silver** | Derived columns are defined, non-overlapping with Bronze |
| **Gold** | All 8 table contracts are defined with required business columns |
| **Config** | Domain values (tiers, statuses, date ranges) are internally consistent |

In [ ]:
from datetime import date
from data_generator.config import DataVolumeConfig, ZomatoDataConfig
from data_generator.generators import (
    generate_customers, generate_restaurants, generate_menu_items,
    generate_orders, generate_deliveries, generate_reviews,
)

volume = DataVolumeConfig(
    num_customers=50, num_restaurants=10, num_orders=200,
    num_deliveries=150, num_reviews=100, num_menu_items_per_restaurant=10,
)
cfg = ZomatoDataConfig()

customers   = generate_customers(volume, cfg)
restaurants = generate_restaurants(volume, cfg)
menu_items  = generate_menu_items(restaurants, volume, cfg)
orders, _   = generate_orders(customers, restaurants, menu_items, volume, cfg)
deliveries  = generate_deliveries(orders, cfg)
reviews     = generate_reviews(orders, volume, cfg)

PASS = "\u2713 PASS"
FAIL = "\u2717 FAIL"
results = []

def check(name, condition):
    status = PASS if condition else FAIL
    results.append((name, status))
    print(f"{status}  {name}")

print("Data generated.")

## Bronze Schema Contracts

Every column listed here must exist in the generator output. Any missing column will break Bronze ingestion.

In [ ]:
BRONZE_CUSTOMER = {
    "customer_id", "first_name", "last_name", "email", "phone",
    "city", "locality", "pincode", "latitude", "longitude",
    "signup_date", "subscription_tier", "is_active", "preferred_payment",
    "preferred_cuisine", "total_orders_lifetime", "avg_order_value",
    "_ingested_at", "_source_system",
}
BRONZE_RESTAURANT = {
    "restaurant_id", "name", "city", "locality", "address", "pincode",
    "latitude", "longitude", "cuisines", "restaurant_type", "avg_cost_for_two",
    "rating", "total_reviews", "is_delivering_now", "has_online_delivery",
    "has_table_booking", "is_premium_partner", "onboarded_date",
    "owner_name", "owner_phone", "gst_number", "_ingested_at", "_source_system",
}
BRONZE_ORDER = {
    "order_id", "customer_id", "restaurant_id", "order_placed_at",
    "order_status", "payment_method", "payment_status", "platform",
    "subtotal", "discount_pct", "discount_amount", "coupon_code",
    "delivery_fee", "tax_amount", "total_amount", "num_items",
    "special_instructions", "_ingested_at", "_source_system",
}
BRONZE_DELIVERY = {
    "delivery_id", "order_id", "delivery_partner_id", "delivery_partner_name",
    "partner_phone", "vehicle_type", "pickup_timestamp", "delivered_timestamp",
    "delivery_time_mins", "delivery_status", "delivery_distance_km",
    "delivery_rating", "_ingested_at", "_source_system",
}
BRONZE_REVIEW = {
    "review_id", "order_id", "customer_id", "restaurant_id",
    "rating", "review_text", "review_timestamp", "sentiment_label",
    "is_verified_order", "upvotes", "has_photo", "_ingested_at", "_source_system",
}

actual_customer    = set(customers[0].keys())
actual_restaurant  = set(restaurants[0].keys())
actual_order       = set(orders[0].keys())
actual_delivery    = set(deliveries[0].keys())
actual_review      = set(reviews[0].keys())

check("Bronze Customers — no missing columns",    not (BRONZE_CUSTOMER - actual_customer))
check("Bronze Customers — no undocumented cols",  not (actual_customer - BRONZE_CUSTOMER))
check("Bronze Restaurants — no missing columns",  not (BRONZE_RESTAURANT - actual_restaurant))
check("Bronze Restaurants — no undocumented cols",not (actual_restaurant - BRONZE_RESTAURANT))
check("Bronze Orders — no missing columns",       not (BRONZE_ORDER - actual_order))
check("Bronze Orders — no undocumented cols",     not (actual_order - BRONZE_ORDER))
check("Bronze Deliveries — no missing columns",   not (BRONZE_DELIVERY - actual_delivery))
check("Bronze Deliveries — no undocumented cols", not (actual_delivery - BRONZE_DELIVERY))
check("Bronze Reviews — no missing columns",      not (BRONZE_REVIEW - actual_review))
check("Bronze Reviews — no undocumented cols",    not (actual_review - BRONZE_REVIEW))
check("All entities have _source_system",         all('_source_system' in e for e in [customers[0], restaurants[0], orders[0], deliveries[0], reviews[0]]))
check("All entities have _ingested_at",           all('_ingested_at' in e for e in [customers[0], restaurants[0], orders[0], deliveries[0], reviews[0]]))

## Silver Schema Contracts

Silver adds derived columns on top of Bronze. These must be defined, non-empty, and non-overlapping with Bronze.

In [ ]:
SILVER_CUSTOMER_DERIVED  = {"full_name", "signup_year", "signup_month", "customer_segment", "_silver_loaded_at"}
SILVER_ORDER_DERIVED     = {"order_date", "order_year", "order_month", "order_day_of_week", "order_hour",
                             "order_time_slot", "is_delivered", "is_cancelled", "has_discount", "_silver_loaded_at"}
SILVER_DELIVERY_DERIVED  = {"delivery_sla_status", "distance_bucket", "_silver_loaded_at"}
SILVER_RESTAURANT_DERIVED= {"price_tier", "rating_tier", "num_cuisines", "_silver_loaded_at"}
SILVER_REVIEW_DERIVED    = {"review_date", "review_text_length", "_silver_loaded_at"}

check("Silver Customer derived cols defined",      len(SILVER_CUSTOMER_DERIVED) >= 4)
check("Silver Order derived cols defined",         len(SILVER_ORDER_DERIVED) >= 8)
check("Silver Delivery derived cols defined",      len(SILVER_DELIVERY_DERIVED) >= 2)
check("Silver Restaurant derived cols defined",    len(SILVER_RESTAURANT_DERIVED) >= 3)
check("Silver Review derived cols defined",        len(SILVER_REVIEW_DERIVED) >= 2)
check("Silver Customer — no Bronze overlap",       not (SILVER_CUSTOMER_DERIVED & BRONZE_CUSTOMER))
check("Silver Order — no Bronze overlap",          not (SILVER_ORDER_DERIVED & BRONZE_ORDER))
check("Silver Delivery — no Bronze overlap",       not (SILVER_DELIVERY_DERIVED & BRONZE_DELIVERY))
check("Silver Restaurant — no Bronze overlap",     not (SILVER_RESTAURANT_DERIVED & BRONZE_RESTAURANT))
check("Silver Review — no Bronze overlap",         not (SILVER_REVIEW_DERIVED & BRONZE_REVIEW))
check("Silver — customer_segment defined",         "customer_segment" in SILVER_CUSTOMER_DERIVED)
check("Silver — order_time_slot defined",          "order_time_slot" in SILVER_ORDER_DERIVED)
check("Silver — delivery_sla_status defined",      "delivery_sla_status" in SILVER_DELIVERY_DERIVED)
check("Silver — price_tier defined",               "price_tier" in SILVER_RESTAURANT_DERIVED)
check("Silver — rating_tier defined",              "rating_tier" in SILVER_RESTAURANT_DERIVED)
check("All Silver layers have _silver_loaded_at",  all('_silver_loaded_at' in s for s in [
    SILVER_CUSTOMER_DERIVED, SILVER_ORDER_DERIVED, SILVER_DELIVERY_DERIVED,
    SILVER_RESTAURANT_DERIVED, SILVER_REVIEW_DERIVED
]))

## Gold Schema Contracts

Validates that all 8 documented Gold tables have their minimum required business columns.

In [ ]:
GOLD_DIM_CUSTOMERS    = {"customer_id", "full_name", "city", "subscription_tier", "customer_segment",
                          "total_orders", "lifetime_revenue", "avg_order_value", "rfm_segment"}
GOLD_DIM_RESTAURANTS  = {"restaurant_id", "name", "city", "rating", "price_tier",
                          "total_orders_received", "total_gmv", "avg_delivery_time_mins", "restaurant_health_score"}
GOLD_FACT_ORDERS      = {"order_id", "customer_id", "restaurant_id", "order_date", "order_status", "total_amount"}
GOLD_AGG_DAILY_CITY   = {"customer_city", "order_date", "total_orders", "total_gmv", "unique_customers", "avg_order_value"}
GOLD_AGG_REVENUE      = {"order_year", "order_month", "total_orders", "total_gmv", "unique_customers",
                          "avg_order_value", "net_revenue"}
GOLD_AGG_REST_PERF    = {"restaurant_id", "name", "city", "restaurant_health_score",
                          "fulfillment_rate_pct", "sla_compliance_pct", "performance_tier"}
GOLD_AGG_DELIVERY_SLA = {"city", "vehicle_type", "delivery_sla_status", "delivery_count", "avg_delivery_mins"}
GOLD_AGG_COHORTS      = {"cohort_month", "activity_month", "active_customers", "total_orders", "cohort_revenue"}

gold_tables = [GOLD_DIM_CUSTOMERS, GOLD_DIM_RESTAURANTS, GOLD_FACT_ORDERS, GOLD_AGG_DAILY_CITY,
               GOLD_AGG_REVENUE, GOLD_AGG_REST_PERF, GOLD_AGG_DELIVERY_SLA, GOLD_AGG_COHORTS]

check("Gold — exactly 8 tables defined",           len(gold_tables) == 8)
check("Gold dim_customers — rfm_segment present",  "rfm_segment" in GOLD_DIM_CUSTOMERS)
check("Gold dim_customers — min 8 columns",        len(GOLD_DIM_CUSTOMERS) >= 8)
check("Gold dim_restaurants — health_score",       "restaurant_health_score" in GOLD_DIM_RESTAURANTS)
check("Gold dim_restaurants — min 8 columns",      len(GOLD_DIM_RESTAURANTS) >= 8)
check("Gold fact_orders — customer FK",            "customer_id" in GOLD_FACT_ORDERS)
check("Gold fact_orders — restaurant FK",          "restaurant_id" in GOLD_FACT_ORDERS)
check("Gold agg_revenue — net_revenue",            "net_revenue" in GOLD_AGG_REVENUE)
check("Gold agg_rest_perf — performance_tier",     "performance_tier" in GOLD_AGG_REST_PERF)
check("Gold agg_delivery_sla — sla_status",        "delivery_sla_status" in GOLD_AGG_DELIVERY_SLA)
check("Gold cohorts — cohort & activity month",    "cohort_month" in GOLD_AGG_COHORTS and "activity_month" in GOLD_AGG_COHORTS)
check("Gold — no table has duplicate columns",     all(len(t) == len(set(t)) for t in gold_tables))

## Pipeline Config Schema Tests

In [ ]:
start = date.fromisoformat(cfg.data_start_date)
end   = date.fromisoformat(cfg.data_end_date)

check("Config — 15 cities",                  len(cfg.cities) == 15)
check("Config — 4 subscription tiers",       set(cfg.subscription_tiers) == {'Free','Zomato Pro','Zomato Pro Plus','Zomato Gold'})
check("Config — 7 order statuses",           set(cfg.order_statuses) == {'Placed','Confirmed','Preparing','Out for Delivery','Delivered','Cancelled','Refunded'})
check("Config — 6 delivery statuses",        set(cfg.delivery_statuses) == {'Assigned','Picked Up','In Transit','Delivered','Failed','Returned'})
check("Config — 6 payment methods",          len(cfg.payment_methods) == 6)
check("Config — 4 order platforms",          len(cfg.order_platforms) == 4)
check("Config — 20 cuisines",                len(cfg.cuisines) == 20)
check("Config — price range valid",          0 < cfg.min_item_price < cfg.max_item_price)
check("Config — delivery fee range valid",   cfg.min_delivery_fee >= 0 and cfg.max_delivery_fee > cfg.min_delivery_fee)
check("Config — date range valid",           start < end)
check("Config — discount probability 0-1",   0.0 <= cfg.discount_probability <= 1.0)
check("Config — max discount <= 100",        0.0 < cfg.max_discount_pct <= 100.0)

## Summary

In [ ]:
passed = sum(1 for _, s in results if s == PASS)
failed = sum(1 for _, s in results if s == FAIL)

print(f"\n{'='*55}")
print(f"  Schema Validation Test Results")
print(f"{'='*55}")
print(f"  Total  : {len(results)}")
print(f"  Passed : {passed}")
print(f"  Failed : {failed}")
print(f"  Result : {'ALL TESTS PASSED' if failed == 0 else f'{failed} TEST(S) FAILED'}")
print(f"{'='*55}")

if failed > 0:
    for name, status in results:
        if status == FAIL:
            print(f"  {FAIL}  {name}")
    raise AssertionError(f"{failed} schema test(s) failed.")